In [4]:
import os
import json

def map_images_to_logs(json_log_path, image_folder, output_path):
    with open(json_log_path, "r", encoding="utf-8") as f:
        logs = json.load(f)

    image_files = sorted(
        [int(f.split("_")[-1].split(".")[0]) for f in os.listdir(image_folder) if f.startswith("PlayerID_")]
    )

    last_available_image = None
    image_dict = {} 

    for log in logs:
        log_id = log["id"]

        if log_id in image_files:
            last_available_image = log_id

        log["image"] = f"./{image_folder}/PlayerID_{last_available_image}.png" if last_available_image else None

        image_dict[log["image"]] = log

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(list(image_dict.values()), f, ensure_ascii=False, indent=4)

json_log_path = "game_log.json"
image_folder = "Players_images"
output_path = "output.json"

map_images_to_logs(json_log_path, image_folder, output_path)


In [5]:
import os
import json
import pandas as pd
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from PIL import Image

with open("output.json", "r", encoding="utf-8") as file:
    data = json.load(file)

wb = Workbook()
ws = wb.active
ws.title = "Players Data"

ws.append(["ID", "Name", "MSSV", "Score", "Timestamp", "Image"])

for index, row in enumerate(data, start=2): 
    ws[f"A{index}"] = row["id"]
    ws[f"B{index}"] = row["name"]
    ws[f"C{index}"] = row["mssv"]
    ws[f"D{index}"] = row["score"]
    ws[f"E{index}"] = row["timestamp"]

    image_path = row["image"]

    if os.path.exists(image_path): 
        try:
            img = Image.open(image_path)
            img.thumbnail((240, 240)) 
            temp_path = f"temp_{index}.png"
            img.save(temp_path)  

            excel_img = XLImage(temp_path)
            ws.add_image(excel_img, f"E{index}") 
        except Exception as e:
            print(f"Lỗi khi xử lý ảnh {image_path}: {e}")
    else:
        ws[f"E{index}"] = "Ảnh không tồn tại"

wb.save("output.xlsx")
